In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg import Connection
from psycopg.rows import dict_row

# 从 Notebook 目录或仓库根目录定位 agent-service。
cwd = Path.cwd()
service_root = next(
    (
        candidate
        for parent in (cwd, *cwd.parents)
        for candidate in (parent, parent / "agent-service")
        if (candidate / "app" / "core" / "config.py").is_file()
    ),
    None,
)
if service_root is None:
    raise RuntimeError("找不到 agent-service，请从项目目录打开 Notebook。")
if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))

load_dotenv(service_root / ".env")
from app.core.config import settings

# Windows Jupyter 的 ProactorEventLoop 与 Psycopg 异步连接不兼容。
# 连接保留给后续单元使用，重跑本单元时关闭旧连接。
if "checkpoint_connection" in globals():
    checkpoint_connection.close()

checkpoint_connection = Connection.connect(
    settings.db.checkpoint_url,
    connect_timeout=10,
    autocommit=True,
    prepare_threshold=0,
    row_factory=dict_row,
)
try:
    checkpointer = PostgresSaver(checkpoint_connection)
    checkpointer.setup()
except BaseException:
    checkpoint_connection.close()
    raise

print("PostgreSQL Checkpointer 表已就绪。")

PostgreSQL Checkpointer 表已就绪。


D:\Profession\Project\hm-insurance\agent-service\app\core\config.py:59: UserWarning: `url` overrides an existing Pydantic `@computed_field` decorator
  def url(self) -> str:


In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage

# 2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 3.定义工具


# 4.创建Agent，绑定模型和工具，Checkpointer
agent = create_agent(
    model = model,
    tools=[],
    checkpointer=checkpointer
)

In [3]:
# 5.调用Agent，要能查询记忆，必须携带thread_id
config = {"configurable": {"thread_id": "thread_01"}}

response = agent.invoke(
    {"messages": [HumanMessage("12个苹果分给4个人，每人分几个？")]},
    config=config,
)
print(response['messages'][-1].content)

这是一个很简单的除法问题：

12 ÷ 4 = 3

所以，每人分 **3个苹果**。


In [4]:
response = agent.invoke(
    {"messages": [HumanMessage("如果是3个人呢？")]},
    config=config,
)
print(response['messages'][-1].content)

如果是分给3个人：

12 ÷ 3 = 4

所以，每人分 **4个苹果**。


In [5]:
# 查询会话历史
state_snapshot = agent.get_state(config)
print(state_snapshot)

StateSnapshot(values={'messages': [HumanMessage(content='12个苹果分给4个人，每人分几个？', additional_kwargs={}, response_metadata={}, id='352e5b63-6052-4f76-86ae-4146377022d2'), AIMessage(content='这是一个很简单的除法问题：\n\n12 ÷ 4 = 3\n\n所以，每人分 **3个苹果**。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 16, 'total_tokens': 40, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'f6b2405c-2313-4e59-a50e-8cd67d50140f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1fce0-1d10-7221-9695-f8f587ada2f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 24, 'total_tokens': 40, 'input_token

In [6]:
{"role": "", "content": ""}
# SystemMessage -> system
# HumanMessage -> user
# AIMessage -> assistant
# ToolMessage -> tool


{'role': '', 'content': ''}

In [7]:
from langchain.messages import HumanMessage, AIMessage

messages = []

for message in state_snapshot.values['messages']:
    # 判断类型，决定role
    if isinstance(message, HumanMessage):
        role = "user"
    elif isinstance(message, AIMessage):
        role = "assistant"
    else:
        continue

    messages.append({"role": role, "content": message.text})

print(messages)

[{'role': 'user', 'content': '12个苹果分给4个人，每人分几个？'}, {'role': 'assistant', 'content': '这是一个很简单的除法问题：\n\n12 ÷ 4 = 3\n\n所以，每人分 **3个苹果**。'}, {'role': 'user', 'content': '如果是3个人呢？'}, {'role': 'assistant', 'content': '如果是分给3个人：\n\n12 ÷ 3 = 4\n\n所以，每人分 **4个苹果**。'}]


In [12]:
checkpointer.delete_thread("cd6c8b2a-7bed-4f99-847f-41e6a1eedf7f")

In [9]:
state_snapshot = agent.get_state(config)
print(state_snapshot)

StateSnapshot(values={'messages': [HumanMessage(content='12个苹果分给4个人，每人分几个？', additional_kwargs={}, response_metadata={}, id='352e5b63-6052-4f76-86ae-4146377022d2'), AIMessage(content='这是一个很简单的除法问题：\n\n12 ÷ 4 = 3\n\n所以，每人分 **3个苹果**。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 16, 'total_tokens': 40, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'f6b2405c-2313-4e59-a50e-8cd67d50140f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1fce0-1d10-7221-9695-f8f587ada2f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 24, 'total_tokens': 40, 'input_token

In [10]:
config = {"configurable": {"thread_id": "de55b937-1265-4122-961a-c22accc07ab3"}}
state_snapshot = agent.get_state(config)
print(state_snapshot)

StateSnapshot(values={}, next=(), config={'recursion_limit': 9999, 'metadata': {'ls_integration': 'langchain_create_agent'}, 'configurable': {'thread_id': 'de55b937-1265-4122-961a-c22accc07ab3'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())
